In [1]:
import tqdm
import pickle
import warnings
import numpy as np
import pandas as pd
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from data import *
from model import *
from utils import *
from recourse_model import LAROAR, ROAR, RecourseCost

warnings.filterwarnings('ignore')

In [2]:
def append_res(d, alg, seed, alpha, lamb, i, x_0, theta_0, beta, x_r, theta_r, p, theta_p, J_r, J_c, robustness, consistency):
    d['alg'].append(alg)
    d['seed'].append(seed)
    d['alpha'].append(alpha)
    d['lambda'].append(lamb)
    d['i'].append(i)
    d['x_0'].append(x_0.round(4))
    d['theta_0'].append(theta_0.round(4))
    d['beta'].append(beta)
    d['x_r'].append(x_r.round(4))
    d['theta_r'].append(theta_r.round(4))
    d['p'].append(p)
    d['theta_p'].append(theta_p.round(4))
    d['J_r'].append(J_r)
    d['J_c'].append(J_c)
    d['robustness'].append(robustness)
    d['consistency'].append(consistency)

In [3]:
def recourse_model_runner(seed, X: np.ndarray, recourse_model1: LAROAR, recourse_model2: ROAR, predictions: List, X_train: np.ndarray, base_model: LR | NN, params: dict):    
    rng = np.random.default_rng(0)
    alpha = recourse_model1.alpha
    lamb = recourse_model1.lamb
    
    results_opt = {'alg': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'theta_0': [], 'beta': [], 'x_r': [], 'theta_r': [], 'p': [], 'theta_p': [], 'J_r': [], 'J_c': [], 'robustness': [], 'consistency': []}
    results_roar = deepcopy(results_opt)
    
    n = len(X)
    for i in tqdm.trange(n, desc=f'Eval alpha={alpha}; lambda={lamb}', colour='#0091ff'):
        x_0 = X[i]
        J = RecourseCost(x_0, lamb)
        
        # lime approximation of original NN
        np.random.seed(i)
        weights_0, bias_0 = lime_explanation(base_model.predict, X_train, x_0)
        weights_0, bias_0 = np.round(weights_0, 4), np.round(bias_0, 4)
        theta_0 = np.hstack((weights_0, bias_0))
        
        recourse_model1.weights = weights_0
        recourse_model1.bias = bias_0
        
        recourse_model2.set_weights(weights_0)
        recourse_model2.set_bias(bias_0)
        
        # robust recourse
        x_r = recourse_model1.get_recourse(x_0, beta=1.)
        weights_r, bias_r = recourse_model1.calc_theta_adv(x_r)
        theta_r = np.hstack((weights_r, bias_r))
        opt_rob = J(x_r, weights_r, bias_r)
            
        # predictions
        if params['data'] == 'temporal':
            theta_r1 = deepcopy(theta_r) * 0.3
            theta_r2 = deepcopy(theta_r) * 0.5
            
            alphas1 = theta_r1 - theta_0
            theta_p1 = theta_0 - alphas1
            
            alphas2 = theta_r2 - theta_0
            theta_p2 = theta_0 - alphas2
        else:
            theta_r1 = deepcopy(theta_r)
            theta_r1[0] = theta_0[0]
            theta_r2 = deepcopy(theta_r) * 0.9
            
            alphas1 = theta_r1 - theta_0
            theta_p1 = theta_0 - alphas1
            
            alphas2 = theta_r2 - theta_0
            theta_p2 = theta_0 - alphas2
        
        predictions = [theta_0, theta_r1, theta_r2, theta_p1, theta_p2]
        
        for p, prediction in enumerate(predictions):
            weights_p, bias_p = prediction[:-1], prediction[[-1]]
            theta_p = (weights_p, bias_p)
            # consistent recourse
            x_c = recourse_model1.get_recourse(x_0, beta=0., theta_p=theta_p)
            opt_con = J(x_c, weights_p, bias_p)
            
            # OPT
            betas = np.arange(0., 1.01, 0.01).round(2)
            alg_num = 0
            for beta in betas:
                x = recourse_model1.get_recourse(x_0, beta=beta, theta_p=theta_p)
                weights_r, bias_r = recourse_model1.calc_theta_adv(x)
                theta_r = np.hstack((weights_r, bias_r))
                
                J_r = J(x, weights_r, bias_r)
                J_c = J(x, weights_p, bias_p)
                rob = J_r - opt_rob
                con = J_c - opt_con
                
                append_res(results_opt, 'OPT', seed, alpha, lamb, i, x_0, theta_0, beta, x, theta_r, p, np.hstack(theta_p), J_r[0], J_c[0], rob[0], con[0])
                
                # convex combination
                for c in np.arange(0., 1.01, 0.01).round(2):
                    x_b = c*x + (1-c)*x_c
                    
                    weights_r, bias_r = recourse_model1.calc_theta_adv(x_b)
                    theta_r = np.hstack((weights_r, bias_r))
                    
                    J_r = J(x_b, weights_r, bias_r)
                    J_c = J(x_b, weights_p, bias_p)
                    rob = J_r - opt_rob
                    con = J_c - opt_con
                    
                    append_res(results_opt, f'OPT{alg_num}', seed, alpha, lamb, i, x_0, theta_0, c, x_b, theta_r, p, np.hstack(theta_p), J_r[0], J_c[0], rob[0], con[0])
                    
                    x_b = c*x_r + (1-c)*x
                    weights_r, bias_r = recourse_model1.calc_theta_adv(x_b)
                    theta_r = np.hstack((weights_r, bias_r))
                    
                    J_r = J(x_b, weights_r, bias_r)
                    J_c = J(x_b, weights_p, bias_p)
                    rob = J_r - opt_rob
                    con = J_c - opt_con
                    
                    append_res(results_opt, f'OPT{alg_num+1}', seed, alpha, lamb, i, x_0, theta_0, c, x_b, theta_r, p, np.hstack(theta_p), J_r[0], J_c[0], rob[0], con[0])
                
                alg_num += 2
                
            # ROAR
            x_r2, _ = recourse_model2.get_recourse(x_0)
            weights_r2, bias_r2 = recourse_model1.calc_theta_adv(x_r2)
            theta_r2 = np.hstack((weights_r2, bias_r2))
            J_r2 = J(x_r2, weights_r2, bias_r2)
            J_c2 = J(x_r2, weights_p, bias_p)
            rob2 = J_r2 - opt_rob
            con2 = J_c2 - opt_con
            
            append_res(results_roar, 'ROAR', seed, alpha, lamb, i, x_0, theta_0, 1., x_r2, theta_r2, p, np.hstack(theta_p), J_r2[0], J_c2[0], rob2[0], con2[0])
        
    df_hist = pd.concat((pd.DataFrame(results_opt), pd.DataFrame(results_roar)))
    df_results = df_hist.groupby(['alg', 'beta', 'p'], as_index=False).mean(True)
        
    return df_results

In [4]:
def recourse_tradeoff(params: dict, seeds, results):
    if params['data'] in ['correction', 'german']:
        data_model = CorrectionShift("../datasets/german.csv", "../datasets/corrected_german.csv", seed=0)
    elif params['data'] in ['temporal', 'business']:
        data_model = TemporalShift("../datasets/SBAcase.11.13.17.csv", seed=0)
    elif params['data'] in ['geospatial', 'student']:
        data_model = GeospatialShift("../datasets/student-por.csv", seed=0)
    elif params['data'] in ['synthetic', 'simulated']:
        data_model = SyntheticData(alpha=1.5, beta=0, n=1000, seed=0)
    
    alpha = params['alpha']
    
    generator = torch.Generator().manual_seed(0)
    predictions = []
    for seed in seeds:
        data1, data2 = data_model.get_data(seed)
        X1_train, y1_train, X1_test, y1_test = data1

        base_model = NN(X1_train.shape[1])
        base_model.train(X1_train.values, y1_train.values)
        
        if seed == 0:
            for _ in range(5):
                prediction = deepcopy(base_model)
                for p, module in enumerate(prediction.model):
                    if hasattr(module, 'weight'):
                        module.weight.data += 0.07*torch.randint(-1, 2, module.weight.data.shape, generator=generator)
                    if hasattr(module, 'bias'):
                        module.bias.data += 0.07*torch.randint(-1, 2, module.bias.data.shape, generator=generator)
                predictions.append(prediction)
        
        recourse_needed_X1_train = recourse_needed(base_model.predict, X1_train.values)
        recourse_needed_X1_test = recourse_needed(base_model.predict, X1_test.values)
            
        weights, bias = None, None
        
        recourse_model1 = LAROAR(
            weights = weights,
            bias = bias,
            alpha = alpha,
        )
    
        recourse_model2 = ROAR(
            weights = weights,
            bias = bias,
            alpha = alpha,
        )
        
        lamb = recourse_model1.choose_lambda(recourse_needed_X1_train, base_model.predict, X1_train.values)
        recourse_model1.lamb = lamb
        recourse_model2.lamb = lamb
        
        df_results = recourse_model_runner(seed, recourse_needed_X1_test, recourse_model1, recourse_model2, predictions, X1_train.values, base_model)
        results.append(df_results)

In [5]:
torch.manual_seed(0)
params = {}
# 'synthetic/simulated', 'correction/german', 'temporal/business', 'geospatial/student'
params['data'] = 'temporal'
# 'lr', 'nn
params['base_model'] = 'nn'
params['alpha'] = 0.5

seeds = range(5)

results = []
recourse_tradeoff(params, seeds, results)
df_results = pd.concat(results)

Eval alpha=0.5; lambda=0.6: 100%|██████████| 38/38 [14:54<00:00, 23.55s/it]


In [9]:
df_results.to_pickle(f'../results/tradeoff_{params["base_model"]}_{params["data"]}.pkl')

In [6]:
df_results_avg = df_results.groupby(['alg', 'beta', 'p'], as_index=False).mean()
df_results_avg[['robustness_std', 'consistency_std']] = df_results.groupby(['alg', 'beta', 'p'], as_index=False).std(ddof=0, numeric_only=True)[['robustness', 'consistency']]
df_results_avg = df_results_avg.sort_values(['alg', 'p', 'beta'])

In [7]:
df_im = deepcopy(df_results_avg)
df_im['alg'] = df_im['alg'].apply(lambda x: 'OPT' if x not in ['OPT', 'ROAR'] else x)

In [8]:
base_model = params['base_model']
data = params['data']

colors = px.colors.qualitative.Plotly
nc = len(colors)
band_colors = [hex2rgba(c, 0.2) for c in colors]
font_family = 'Times New Roman'
font_color = 'black'
width, height = 720, 540
symbols = ['circle', 'square', 'triangle-up', 'cross', 'x']
show_errors = False

fig = go.Figure()
for p in df_im['p'].unique():
    for ii, alg in enumerate(df_im['alg'].unique()):
        temp = df_im[(df_im['p'] == p) & (df_im['alg'] == alg)].sort_values(['consistency'], ascending=False)
        mask = pareto_frontier(temp['robustness'], temp['consistency'])
        temp = temp.iloc[mask]
        t = '\\theta_{p}'.format(p=f'{p}')
        if alg in df_im['alg'].unique():
            if alg == 'ROAR':
                name = f'({temp["consistency"].round(2).iloc[0]:.2f}, {temp["robustness"].round(2).iloc[0]:.2f})'
            else:
                if base_model == 'lr':
                    name = r'$\hat{theta}$'.format(theta=t) if p != 0 else r'$\theta_{0}$'
                else:
                    name = r'$\hat{theta}$'.format(theta=t)

            fig.add_trace(go.Scatter(
                # comment this x, y out if you don't want to see the ROAR points
                # x = temp['consistency'] if alg != 'ROAR' else [None],
                # y = temp['robustness'] if alg != 'ROAR' else [None],
                x = temp['consistency'],
                y = temp['robustness'],
                marker = dict(color=colors[p] if alg in ['OPT', 'ROAR'] else colors[(p+ii)%nc], symbol=symbols[p] if alg != 'ROAR' else 'star', size=2 if alg != 'ROAR' else 10),
                mode = 'markers+lines' if alg != 'ROAR' else 'markers',
                name = name,
                showlegend=True,
                customdata=temp['beta'],
                hovertemplate='consistency: %{x}<br>robustness: %{y}<br>beta: %{customdata}'
            ))

            if show_errors:
                if alg != 'ROAR':

                    fig.add_trace(go.Scatter(
                        x = temp['consistency'],
                        y=temp['robustness'] + temp['robustness_std'],
                        marker = dict(color=colors[p]),
                        mode='lines',
                        line=dict(width=0),
                        showlegend=False
                    ))
                    
                    fig.add_trace(go.Scatter(
                        x = temp['consistency'],
                        y=temp['robustness'] - temp['robustness_std'],
                        marker = dict(color=colors[p]),
                        mode='lines',
                        line=dict(width=0),
                        fillcolor=band_colors[p],
                        fill='tonexty',
                        showlegend=False
                    ))

fig.update_xaxes(
    title=dict(
        text='Consistency',
        font=dict(
            family=font_family,
            color=font_color,
            size=25
        )
        ), 
    showline=True, 
    mirror=True,
    linecolor='black', 
    gridcolor='lightgrey', 
    zerolinewidth=1,
    zerolinecolor='lightgrey',
    )


fig.update_yaxes(
    title=dict(
        text='Robustness',
        font=dict(
            family=font_family,
            color=font_color,
            size=25
        ), 
        ), 
    showline=True, 
    mirror=True,
    linecolor='black', 
    gridcolor='lightgrey',
    zerolinewidth=1,
    zerolinecolor='lightgrey',
    )


fig.update_layout(
    legend=dict(
        x=0.975, 
        y=0.975, 
        xanchor='right',
        font=dict(
            family=font_family,
            color=font_color,
            size=15
            ), 
        bgcolor='rgba(255, 255, 255, 0.7)',
        bordercolor='lightgrey',
        borderwidth=1,
        entrywidth=0.1,
        entrywidthmode='pixels',
        ),
    width=width,
    height=height,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickfont=dict(
            family=font_family,
            color=font_color,
            size=20,
        )
    ),
    yaxis=dict(
        tickfont=dict(
            family=font_family,
            color=font_color,
            size=20
        )
    )
    )

fig.show()